Here’s a comprehensive list of **classification metrics**, organized by category. These are commonly used for evaluating the performance of classification models (binary, multiclass, and multilabel):

---

## 🔹 **Threshold-Based Metrics (using confusion matrix)**

* **Accuracy**
* **Precision (Positive Predictive Value)**
* **Recall (Sensitivity, True Positive Rate)**
* **Specificity (True Negative Rate)**
* **F1 Score**
* **Fβ Score** (generalized F1)
* **False Positive Rate (FPR)**
* **False Negative Rate (FNR)**
* **Matthews Correlation Coefficient (MCC)**
* **Balanced Accuracy**
* **Cohen’s Kappa**
* **Jaccard Index (Intersection over Union)**
* **Hamming Loss**
* **Zero-One Loss**
* **Youden’s J Statistic** (`sensitivity + specificity - 1`)

---

## 🔹 **Probability-Based Metrics**

* **Log Loss (Cross-Entropy Loss)**
* **Brier Score**
* **Calibration Error**

  * Expected Calibration Error (ECE)
  * Maximum Calibration Error (MCE)

---

## 🔹 **Ranking / Threshold-Independent Metrics**

* **ROC AUC (Area Under the Receiver Operating Characteristic Curve)**
* **Precision-Recall AUC (PR AUC)**
* **Average Precision (AP)**
* **Lift**
* **Gain**
* **KS Statistic (Kolmogorov–Smirnov Statistic)**

---

## 🔹 **Multiclass / Multilabel Specific Metrics**

* **Macro / Micro / Weighted Averages** of:

  * Precision
  * Recall
  * F1 Score
* **Top-K Accuracy**
* **Coverage Error**
* **Label Ranking Average Precision (LRAP)**

---

## 🔹 **Other Derived or Domain-Specific Metrics**

* **Fowlkes–Mallows Index**
* **Detection Rate / Miss Rate**
* **True Skill Statistic (TSS)**
* **Threat Score (Critical Success Index)**

---

## ✅ Notes:

* Most metrics can be **macro, micro, or weighted** when applied to multiclass or multilabel classification.
* **Choosing metrics depends** on the problem type (binary, multiclass, multilabel), class balance, and domain requirements.

Let me know if you want definitions or formulas for any of these.


In [2]:

import random
import pandas as pd
import numpy as np
from typing import List, Dict, Tuple, Any
from sklearn.datasets import make_classification, load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score, roc_auc_score, roc_curve, precision_recall_curve
from scipy.stats import ks_2samp
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, log_loss, average_precision_score
)

# Models
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Dataviz
from matplotlib.colors import rgb2hex
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import matplotlib
import seaborn as sns
import plotly.express as px
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.colors import rgb2hex




from sklearn.preprocessing import StandardScaler, LabelEncoder

In [3]:
import sys
import os
from pathlib import Path

path = Path(os.path.abspath(""))
base_path = path.parent.parent.absolute()
telco_path = os.path.join(str(base_path), '1_dataset', 'telco', 'telco_churn_data.csv')

# sys.path.append(os.path.abspath(src_path))

# data_path = os.path.join(str(base_path), 'data')
# data_sfb_path = os.path.join(data_path, 'sao_francisco_basin')

# from series_analysis import perform_stationary_tests, check_seasonality_pd, TrendTypeDetector, granger_causation_matrix, select_optimal_p, create_granger_matrix_from_dict, make_granger_tests
# # from data_processing import make_series_regular, processing_pipeline
# from modeling import create_arima_model, create_train_test_split
# from utils import retain_period_common_to_all
# from visualization import plot_test_comparison, plotly_acf, plot_ts, plotly_pacf
# from networks import BaseNetwork, NetworkGranger, get_network_metrics, GenericNetwork
# from ccm_test import TestCCM
# from granger_test import var_lag_selection_info_criteria, make_granger_tests2
# from cointegration_test import make_cointegration_tests

In [4]:
df = pd.read_csv(telco_path)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')


def augment_dataset_location_random(df, location_list, weights=None):
    _df = df.copy()
    if isinstance(weights, type(None)):
        _df['Location'] = np.random.choice(location_list, size=len(_df), p=weights)
        return _df
    _df['Location'] = np.random.choice(location_list, size=len(_df))
    return _df


def augment_dataset_location_cluster(df, features, n_clusters):
    # Cluster customers and assign each cluster a region:
    _df = df.copy()
    # X = _df[features].fillna(0)
    X = _df[features].fillna(_df[features].mean())
    X_scaled = StandardScaler().fit_transform(X)

    # Cluster into 6 groups
    kmeans = KMeans(n_clusters=n_clusters, random_state=42).fit(X_scaled)
    _df['Cluster'] = kmeans.labels_

    # Map clusters to locations
    cluster_to_location = {
        0: 'Chicago',
        1: 'New York',
        2: 'Boston',
        3: 'Miami',
        4: 'Los Angeles',
        5: 'Stanford',
    }
    _df['Location'] = _df['Cluster'].map(cluster_to_location)
    _df.drop(columns='Cluster', inplace=True)
    return _df


def augment_dataset_location_probabilistic(df, charge_col, config):
    location = np.empty(len(df), dtype=object)

    for level in config.values():
        lower, upper = level['range']
        mask = df[charge_col].between(lower, upper, inclusive='left')
        location[mask] = np.random.choice(
            level['choices'],
            size=mask.sum(),
            p=level['probs']
        )

    return location


def augment_dataset_datetime_random(df, start_date, end_date):
    _df = df.copy()
    n_rows = len(_df)
    random_days = np.random.randint(0, (end_date - start_date).days + 1, size=n_rows)
    _df['Date'] = start_date + pd.to_timedelta(random_days, unit='D')
    return _df

def augment_dataset_datetime_monthend(df, start_date, end_date):
    _df = df.copy()
    n_rows = len(_df)
    month_end_dates = pd.date_range(start=start_date, end=end_date, freq='M').to_list()
    if pd.to_datetime(end_date).day != pd.to_datetime(end_date).daysinmonth:
        if pd.to_datetime(end_date) > month_end_dates[-1]:
            month_end_dates.append(pd.to_datetime(end_date))
    random_month_ends = np.random.choice(month_end_dates, size=n_rows, replace=True)
    _df['Date'] = random_month_ends
    return _df

# Location Augmentation
# weights = [0.2, 0.25, 0.1, 0.15, 0.2, 0.1]
# location_list = ['Chicago', 'New York', 'Boston', 'Miami', 'Los Angeles', 'Stanford']

# features_to_cluster = ['MonthlyCharges', 'tenure', 'TotalCharges']
# n_clusters = 6

location_config = {
    'high': {
        'range': (80, np.inf),  # MonthlyCharges > 80
        'choices': ['New York', 'Los Angeles', 'Miami', 'Chicago', 'Boston', 'Stanford'],
        'probs':   [0.35, 0.35, 0.1, 0.1, 0.05, 0.05]
    },
    'mid': {
        'range': (60, 80),  # 60 < MonthlyCharges <= 80
        'choices': ['Chicago', 'Miami', 'New York', 'Los Angeles', 'Boston', 'Stanford'],
        'probs':   [0.3, 0.3, 0.15, 0.15, 0.05, 0.05]
    },
    'low': {
        'range': (0, 60),  # MonthlyCharges <= 60
        'choices': ['Boston', 'Stanford', 'Chicago', 'Miami', 'New York', 'Los Angeles'],
        'probs':   [0.4, 0.3, 0.1, 0.1, 0.05, 0.05]
    }
}


# Datetime Augmentation
start_date = pd.to_datetime('2023-01-01')
end_date = pd.to_datetime('2023-07-31')



def expand_customer_history(df: pd.DataFrame) -> pd.DataFrame:
    """
    Expands a customer snapshot DataFrame into a historical, month-by-month view.

    For each customer, it generates a record for every month of their tenure,
    back-dating the 'Date' and recalculating 'TotalCharges' accordingly.

    Args:
        df (pd.DataFrame): The input DataFrame containing one row per customer
                           with their final tenure and date.

    Returns:
        pd.DataFrame: An expanded DataFrame with a full monthly history for each customer.
    """
    # Ensure the 'Date' column is a proper datetime object
    df['Date'] = pd.to_datetime(df['Date'])
    
    # List to hold all the new, historical records
    all_records = []

    # Iterate over each customer's final record
    for _, row in df.iterrows():
        final_tenure = row['tenure']
        final_date = row['Date']
        monthly_charges = row['MonthlyCharges']
        
        # Create a record for each month of the customer's tenure
        for i in range(1, final_tenure + 1):
            # Create a copy of the original row to modify
            new_record = row.copy()
            
            # 1. Set the tenure for this historical record
            new_record['tenure'] = i
            
            # 2. Calculate the date for this historical record
            # The date is calculated by offsetting backwards from the final date.
            # For a tenure of 1, the offset is (final_tenure - 1) months.
            months_to_offset = final_tenure - i
            new_record['Date'] = final_date - DateOffset(months=months_to_offset)
            
            # 3. Recalculate TotalCharges for this point in time
            # TotalCharges is simply the monthly charge multiplied by the current tenure.
            new_record['TotalCharges'] = round(monthly_charges * i, 2)
            
            # 4. Adjust the Churn status
            # A customer could only have churned in their final month.
            # All historical records before that must be 'No'.
            if i < final_tenure:
                new_record['Churn'] = 'No'
            
            # Add the newly created historical record to our list
            all_records.append(new_record)

    # Concatenate all records into a final DataFrame and sort for clarity
    expanded_df = pd.DataFrame(all_records)
    expanded_df = expanded_df.sort_values(by=['customerID', 'tenure']).reset_index(drop=True)
    
    return expanded_df


# df = augment_dataset_location_random(df, location_list, weights=weights)
# df = augment_dataset_location_cluster(df, features_to_cluster, n_clusters)
df['Location'] = augment_dataset_location_probabilistic(df, 'MonthlyCharges', location_config)
df = augment_dataset_datetime_monthend(df, start_date, end_date)
# df = expand_customer_history()

features_col = [
    'customerID', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges'
]
target_col = 'Churn'

df.dropna(inplace=True)

def object_to_int(dataframe_series):
    if dataframe_series.dtype=='object':
        dataframe_series = LabelEncoder().fit_transform(dataframe_series)
    return dataframe_series


def object_to_int2(df, feature_list):
    for feat in feature_list:
        if df[feat].dtype == 'object':
            df[feat] = LabelEncoder().fit_transform(df[feat])
    return df


# df = df.apply(lambda x: object_to_int2(x))
df = object_to_int2(df, features_col + [target_col])

df.head()

X_df = df.drop(columns=['Churn'])
y_df = df[['Churn']]



X_train_df, X_test_df, y_train_df, y_test_df = train_test_split(X_df, y_df, test_size=0.30, random_state=40) # , stratify=y


C:\Users\douglas.sgrott_indic\AppData\Local\Temp\ipykernel_31664\3623164758.py:65: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  month_end_dates = pd.date_range(start=start_date, end=end_date, freq='M').to_list()


In [5]:
X_train_df.head(3)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Location,Date
2194,1723,Male,0,0,0,1,Yes,No,Fiber optic,0,...,0,Yes,No,0,1,2,79.5,79.5,Los Angeles,2023-06-30
4117,4396,Male,0,1,0,63,Yes,No,Fiber optic,2,...,0,Yes,Yes,2,0,1,99.7,6330.4,New York,2023-01-31
2403,195,Female,0,1,0,71,Yes,No,DSL,2,...,2,Yes,Yes,1,1,0,84.8,6046.1,Los Angeles,2023-06-30


In [6]:

# # --- Dataset Creation ---
# n_features = 10
# X, y = make_classification(n_samples=1000, n_features=n_features, n_informative=4, n_redundant=4, flip_y=0.2, weights=[0.8, 0.1], random_state=42)
# y = y.reshape(-1, 1)
# feature_names = [f'Feature {i+1}' for i in range(n_features)]
# target_name = ['Target']
# columns = feature_names + target_name
# df = pd.DataFrame(np.concatenate([X, y], axis=1), columns=columns)

# df_train, df_test = train_test_split(df, test_size=None, train_size=None, random_state=None, shuffle=True, stratify=None)
# X_train_df, X_test_df = df_train[feature_names], df_test[feature_names]
# y_train_df, y_test_df = df_train[target_name], df_test[target_name]


# def augment_dataset_location_random(df, location_list, weights=None):
#     if isinstance(weights, type(None)):
#         df['Location'] = np.random.choice(location_list, size=len(df), p=weights)
#         return df
#     df['Location'] = np.random.choice(location_list, size=len(df))
#     return df

# # Location Augmentation
# weights = [0.2, 0.25, 0.1, 0.15, 0.2, 0.1]
# location_list = ['Chicago', 'New York', 'Boston', 'Miami', 'Los Angeles', 'Stanford']
# df = augment_dataset_location_random(df, location_list, weights=weights)

# df.head(5)


In [7]:
model = LogisticRegression()
model.fit(X_train_df[features_col], y_train_df[target_col])

c:\Users\douglas.sgrott_indic\miniconda3\envs\tsfresh_venv\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [18]:

def calculate_ks(model, X, y):
    """
    Calculate the Kolmogorov-Smirnov (KS) statistic and p-value for a given model and dataset.

    Parameters
    ----------
    model : object
        A model with a predict_prob method that returns predicted probabilities.
    X : array-like or DataFrame
        Feature matrix.
    y : array-like or Series
        True labels (0 or 1).

    Returns
    -------
    ks_stat : float
        KS statistic value.
    p_value : float
        Two-tailed p-value associated with the KS statistic.
    """
    # Get predicted probabilities
    predicted_proba = model.predict_proba(X)

    # Separate probabilities by class
    positive_proba = predicted_proba[:,1]#[y == 1]
    negative_proba = predicted_proba[:,0]#[y == 0]

    # Calculate the KS statistic and p-value
    ks_stat, p_value = ks_2samp(positive_proba, negative_proba)

    return ks_stat, p_value


def calculate_model_metrics_datasets(dataset_dict):
    roc_auc, accuracy, f1, precision, recall, ks = [], [], [], [], [], []
    specificity, mcc, logloss, pr_auc, balanced_accuracy = [], [], [], [], []
    for set_name in dataset_dict:
        # x_ = dataset_dict[set_name]['x']
        y_ = dataset_dict[set_name]['y']
        y_pred = dataset_dict[set_name]['y_pred']
        y_prob = dataset_dict[set_name]['y_prob']
        # y_pred_ = model.predict(x_)
        # y_pred_prob = model.predict_proba(x_)[:, 1]  # Assumes binary classification (class 1)

        tn, fp, fn, tp = confusion_matrix(y_, y_pred).ravel()

        roc_auc.append(roc_auc_score(y_, y_pred))
        accuracy.append(accuracy_score(y_, y_pred))
        f1.append(f1_score(y_, y_pred))
        precision.append(precision_score(y_, y_pred))
        recall.append(recall_score(y_, y_pred))
        # ks_stat_value, p = calculate_ks(model, x_, y_)
        # ks.append(ks_stat_value)

        specificity.append(tn / (tn + fp))  # Specificity (True Negative Rate)
        mcc.append((tp * tn - fp * fn) / 
                   ((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) ** 0.5)  # Matthews Correlation Coefficient
        logloss.append(log_loss(y_, y_prob))  # Log Loss
        pr_auc.append(average_precision_score(y_, y_prob))  # Precision-Recall AUC
        balanced_accuracy.append((recall[-1] + specificity[-1]) / 2)  # Balanced Accuracy
    metrics = pd.DataFrame({
        'ROC AUC': roc_auc,
        'Accuracy': accuracy,
        'F1 Score': f1,
        'Precision': precision,
        'Recall': recall,
        # 'KS': ks,
        'Specificity': specificity,
        'MCC': mcc,
        'Log Loss': logloss,
        'PR AUC': pr_auc,
        'Balanced Accuracy': balanced_accuracy,
    }, index=dataset_dict.keys())
    return metrics


dataset_dict = {"Data": {"x": X_df[features_col], "y": y_df}}

dataset_dict = {
    "Data":
    {
        "x": X_df[features_col],
        "y": y_df,
        "y_pred": model.predict(X_df[features_col]),
        "y_prob": model.predict_proba(X_df[features_col])[:, 1],
    }
}

calculate_model_metrics_datasets(dataset_dict)

,ROC AUC,Accuracy,F1 Score,Precision,Recall,Specificity,MCC,Log Loss,PR AUC,Balanced Accuracy
Data,0.721613,0.795222,0.594366,0.627603,0.564473,0.878753,0.45907,0.425794,0.641396,0.721613


## Dataset Partition Metrics (Train-Test)

In [19]:
dataset_dict = {
    "Train":
    {
        "x": X_train_df[features_col],
        "y": y_train_df,
        "y_pred": model.predict(X_train_df[features_col]),
        "y_prob": model.predict_proba(X_train_df[features_col])[:, 1],
    },
    "Test":
    {
        "x": X_test_df[features_col],
        "y": y_test_df,
        "y_pred": model.predict(X_test_df[features_col]),
        "y_prob": model.predict_proba(X_test_df[features_col])[:, 1],
    },
}

calculate_model_metrics_datasets(dataset_dict)


,ROC AUC,Accuracy,F1 Score,Precision,Recall,Specificity,MCC,Log Loss,PR AUC,Balanced Accuracy
Train,0.726995,0.800081,0.603865,0.642123,0.569909,0.884082,0.472302,0.420575,0.660463,0.726995
Test,0.708973,0.783886,0.572233,0.594542,0.551537,0.866410,0.428466,0.437969,0.598235,0.708973


## Other Data Partition Metrics (Custom)

In [20]:

dataset_dict = {
    location:
        {"y": df[df['Location'] == location][target_col],
         "y_pred": model.predict(df[df['Location'] == location][features_col]),
         "y_prob": model.predict_proba(df[df['Location'] == location][features_col])[:, 1],}
        for location in df['Location'].unique()
}

calculate_model_metrics_datasets(dataset_dict)


,ROC AUC,Accuracy,F1 Score,Precision,Recall,Specificity,MCC,Log Loss,PR AUC,Balanced Accuracy
Stanford,0.676344,0.822387,0.505155,0.636364,0.418803,0.933884,0.415566,0.383699,0.601056,0.676344
Boston,0.662107,0.820329,0.469345,0.569231,0.399281,0.924933,0.373507,0.379499,0.531966,0.662107
Miami,0.754288,0.795132,0.654110,0.654110,0.654110,0.854467,0.508576,0.440902,0.692951,0.754288
Los Angeles,0.720986,0.766142,0.615783,0.631300,0.601010,0.840961,0.448127,0.462090,0.667830,0.720986
New York,0.740001,0.787267,0.632708,0.639566,0.625995,0.854007,0.483060,0.447209,0.665014,0.740001
Chicago,0.731241,0.778218,0.617747,0.615646,0.619863,0.842618,0.461549,0.447187,0.645295,0.731241


## Vintage Metrics

In [ ]:

dataset_dict = {
    date:
        {"y": df[df['Date'] == date][target_col],
         "y_pred": model.predict(df[df['Date'] == date][features_col]),
         "y_prob": model.predict_proba(df[df['Date'] == date][features_col])[:, 1],}
        for date in sorted(df['Date'].unique())
}

calculate_model_metrics_datasets(dataset_dict)


,ROC AUC,Accuracy,F1 Score,Precision,Recall,Specificity,MCC,Log Loss,PR AUC,Balanced Accuracy
2023-01-31,0.726180,0.801418,0.603239,0.647826,0.564394,0.887967,0.473635,0.435104,0.669453,0.726180
2023-02-28,0.716094,0.788443,0.590909,0.631579,0.555160,0.877027,0.450736,0.428222,0.645555,0.716094
2023-03-31,0.732856,0.801661,0.606186,0.622881,0.590361,0.875350,0.474070,0.420254,0.629144,0.732856
2023-04-30,0.719355,0.797284,0.595745,0.652542,0.548043,0.890667,0.464966,0.417631,0.665946,0.719355
2023-05-31,0.721549,0.793574,0.593870,0.622490,0.567766,0.875332,0.456748,0.439182,0.636082,0.721549
2023-06-30,0.710118,0.787046,0.565130,0.573171,0.557312,0.862924,0.424240,0.418355,0.605158,0.710118
2023-07-31,0.726789,0.797764,0.605941,0.645570,0.570896,0.882682,0.472216,0.421643,0.661977,0.726789
